In [13]:
:dep base64 = "0.22"
:dep etherparse = "0.14"

use base64::Engine;
use std::fs;
use std::io::{BufReader, Read};

In [15]:
{
    let pcap_path = std::env::args().nth(1).unwrap_or_else(|| "q8.pcap".to_string());
    let file = fs::File::open(&pcap_path).expect("ファイルを開けません");
    let mut reader = BufReader::new(file);

    // グローバルヘッダ(24バイト)をスキップ
    let mut global_header = [0u8; 24];
    reader.read_exact(&mut global_header).unwrap();

    let mut found: Vec<String> = Vec::new();

    // パケットレコードを順番に読む
    loop {
        // パケットヘッダ(16バイト)
        let mut pkt_header = [0u8; 16];
        if reader.read_exact(&mut pkt_header).is_err() { break; }

        let incl_len = u32::from_le_bytes(pkt_header[8..12].try_into().unwrap()) as usize;

        let mut pkt_data = vec![0u8; incl_len];
        if reader.read_exact(&mut pkt_data).is_err() { break; }

        // Ethernet(14) + IP(20) + TCP(20) = 54バイト以上必要
        if pkt_data.len() < 54 { continue; }

        // TCPペイロード開始位置
        // IPヘッダ長を確認
        let ip_ihl = (pkt_data[14] & 0x0F) as usize * 4;
        // TCPデータオフセットを確認
        let tcp_offset = (pkt_data[14 + ip_ihl + 12] >> 4) as usize * 4;
        let payload_start = 14 + ip_ihl + tcp_offset;

        if payload_start >= pkt_data.len() { continue; }

        let payload = &pkt_data[payload_start..];
        let text = match std::str::from_utf8(payload) {
            Ok(s) => s,
            Err(_) => continue,
        };

        if !text.starts_with("GET ") { continue; }

        for line in text.lines() {
            if line.to_lowercase().starts_with("authorization: basic ") {
                let b64 = line["authorization: basic ".len()..].trim();
                println!("Base64: {}", b64);
                if let Ok(decoded) = base64::engine::general_purpose::STANDARD.decode(b64) {
                    let s = String::from_utf8_lossy(&decoded).to_string();
                    found.push(s);
                }
            }
        }
    }

    if found.is_empty() {
        println!("見つかりませんでした");
    } else {
        found.dedup();
        let output = found.join("\n");
        fs::write("flag.txt", &output).unwrap();
        println!("\nflag.txt に保存");
    }
}

Base64: cTg6RkxBR181dXg3eksyTktTSDhmU0dB

flag.txt に保存


()